# Lab 1: 사전 요구 사항 및 인프라 설정

## 개요
워크숍의 모든 사전 요구 사항을 확인하고 AWS에 CRM 애플리케이션 스택(EC2 + NGINX + DynamoDB)을 배포합니다.

## 목표
**Part 1: 사전 요구 사항**
- Python 버전(3.10 이상) 확인
- AWS 계정 및 자격 증명 확인
- 워크숍 종속성 설치
- Bedrock AgentCore SDK 및 starter toolkit 확인
- Bedrock 모델 액세스 테스트
- 공유 컨텍스트를 위한 Agent Memory 설정
- Amazon Cognito를 사용하여 사용자 및 Agent 자격 증명 설정

**Part 2: 인프라 설정**
- AWS 인프라 프로비저닝: EC2 인스턴스, NGINX, DynamoDB, CloudWatch
- 샘플 CRM 애플리케이션 배포
- 장애를 시뮬레이션하는 결함 주입 스크립트 생성
- CloudWatch 모니터링 설정
- 인프라가 실행 중이고 액세스 가능한지 확인

## 학습 내용
- 워크숍 사전 요구 사항 및 설정 워크플로
- 인시던트 대응 테스트를 위한 결함 주입 패턴
- 진단을 위한 CloudWatch 로그 및 지표 설정

## 1. Python 버전 확인

In [ ]:
import sys

print(f"Python version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"
print("✅ Python version check passed")

## 2. 워크숍 종속성 설치

In [ ]:
%pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

## 3. AWS 구성 확인

In [ ]:
import boto3
from lab_helpers.config import AWS_REGION, AWS_PROFILE, MODEL_ID, WORKSHOP_NAME
from lab_helpers.lab_01.infrastructure import get_app_url

# 구성 표시
print(f"Workshop Name: {WORKSHOP_NAME}")
print(f"AWS Region: {AWS_REGION}")
print(f"Model ID: {MODEL_ID}\n")

# AWS 자격 증명 확인
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
sts = session.client("sts")
identity = sts.get_caller_identity()

print(f"✅ AWS Account: {identity['Account']}")
print(f"✅ AWS User/Role: {identity['Arn']}")

## 4. Bedrock 모델 액세스 테스트

In [ ]:
import boto3
from lab_helpers.config import AWS_REGION, MODEL_ID, AWS_PROFILE

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock = session.client("bedrock", region_name=AWS_REGION)

# 모델 액세스 확인
try:
    model = bedrock.get_foundation_model(modelIdentifier=MODEL_ID)
    print(f"Model ID: {MODEL_ID}")
    print("✅ Bedrock model access verified")
except Exception as e:
    print(f"❌ Error accessing model: {e}")
    raise

## 5. AgentCore 구성 요소 확인

In [ ]:
import importlib

packages = ["bedrock_agentcore", "strands", "boto3", "pydantic"]

for package in packages:
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, "__version__", "installed")
        print(f"✅ {package:<20} {version}")
    except ImportError:
        print(f"❌ {package:<20} NOT FOUND")

print("\n✅ All core packages verified")

## 요약
✅ 모든 사전 요구 사항을 확인했습니다. Lab 1: 인프라 설정 및 결함 주입을 진행할 준비가 되었습니다.

## Part 1.5: Cognito 설정(Lab 3-5 인증)

### 개요

이 섹션에서는 Lab 3-5에서 사용하는 인증 인프라를 위해 AWS Cognito를 설정합니다.

**생성할 리소스:**
- Cognito User Pool: `aiml301-UserPool`
- **사용자 그룹 2개**(신규):
  - **developers**: 복구 계획을 생성하는 사용자
  - **approvers**: 계획을 승인하고 실행하는 사용자
- **App Client 2개**:
  - **User Auth Client**(public): OAuth를 지원하는 최종 사용자 인증용
  - **M2M Client**(confidential): Gateway-to-Runtime 서비스 간 인증용
- **Resource Server**: 세분화된 권한 부여를 위한 사용자 지정 범위(`mcp.invoke`, `runtime.access`)
- **User Pool Domain**: OAuth2 토큰 엔드포인트
- **테스트 사용자 2명**:
  - **Developer User**: `testuser@aiml301.example.com`(`developers` 그룹 구성원)
  - **Approver User**: `approver@aiml301.example.com`(`approvers` 그룹 구성원)

**인증 흐름:**
1. **User Auth**(Client → Gateway): 최종 사용자가 자격 증명으로 인증하고 그룹 구성원 정보가 포함된 JWT 토큰을 받습니다.
2. **M2M Auth**(Gateway → Runtime): Gateway가 client credentials grant를 사용하여 Runtime 액세스용 M2M 토큰을 가져옵니다.

**다중 행위자 워크플로(Lab-03):**
- **Developer** 로그인 → 복구 계획 생성 → 차단됨(승인 필요)
- **Approver** 로그인 → 대기 중인 인시던트 검색 → 계획 검토 → 실행 승인
- **Developer** 복귀 → 공유 메모리에서 승인 확인 → 승인된 단계 실행

**JWT 토큰 클레임:**
설정 후 JWT ID 토큰에는 다음 내용이 포함됩니다.
```json
{
  "email": "developer@aiml301.example.com",
  "cognito:username": "developer@aiml301.example.com",
  "cognito:groups": ["developers"],
  "sub": "uuid",
  "scope": "openid profile email custom-scopes"
}
```

**그룹을 사용하는 이유**
- **역할 기반 권한 부여**: Gateway는 실행을 허용하기 전에 사용자가 `approvers` 그룹에 속하는지 확인할 수 있습니다.
- **인시던트 라우팅**: 대기 중인 인시던트는 승인자에게만 알립니다.
- **감사 추적**: Memory 레코드에서 각 작업을 수행한 역할을 확인할 수 있습니다.
- **행위자 식별**: `email` 클레임은 UUID 대신 읽기 쉬운 actor_id를 제공합니다.

### 목표

✅ 중앙 집중식 인증을 위한 Cognito 인프라 설정
✅ 역할 기반 액세스 제어를 위한 사용자 그룹 생성
✅ 사용자 기반 및 서비스 간 이중 인증 모드 활성화
✅ 세분화된 권한 부여 범위 생성
✅ 풍부한 JWT ID 토큰을 위한 OAuth 흐름 활성화
✅ 이후 Lab에서 사용할 구성을 SSM Parameter Store에 저장

#### 1. Cognito 설정 실행

In [ ]:
from lab_helpers.cognito_setup import setup_cognito_complete

# 전체 Cognito 설정 워크플로 실행
cognito_config = setup_cognito_complete()

print("\n" + "=" * 70)
print("COGNITO SETUP COMPLETE")
print("=" * 70)
print("Cognito User Pool ID: ", cognito_config["user_pool_id"])

## Part 1.6: Lab 2-5를 위한 Memory 설정

이 섹션에서는 모든 Agent Lab(2-5)이 대화 기록 및 세션 관리에 사용할 공유 AgentCore Memory 리소스를 생성합니다.

### 생성할 리소스:
- 만료 기간이 7일인 AgentCore Memory 리소스
- Lab 2-5에서 쉽게 액세스할 수 있도록 memory_id를 Parameter Store에 저장
- Lab 2-4의 정적 세션 추적을 위한 기본 세션 ID 저장

### 핵심 학습 내용:
Memory를 사용하면 Agent 호출과 멀티턴 대화 전반에서 컨텍스트를 유지할 수 있습니다. 모든 Lab은 이 단일 Memory 리소스를 공유합니다.

### 목표
✅ AgentCore Memory 리소스 생성  
✅ Memory 구성을 Parameter Store에 저장  
✅ 다운스트림 Agent의 대화 기록 로드 활성화

In [ ]:
### 1.6.1: AgentCore Memory 리소스 생성

from bedrock_agentcore.memory import MemoryClient
from lab_helpers.constants import PARAMETER_PATHS
from datetime import datetime

memory_client = MemoryClient(region_name=AWS_REGION)
memory_name = f"{PARAMETER_PATHS['memory']['memory_name_prefix']}_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Creating memory: {memory_name}")
memory = memory_client.create_memory_and_wait(
    name=memory_name,
    description="SRE Agent Shared Short-Term Memory for Labs 2-5",
    strategies=[],
    event_expiry_days=7,
    max_wait=600,
    poll_interval=10,
)

memory_id = memory["id"]
print(f"✅ Memory created: {memory_id} (Status: ACTIVE, Expiry: 7 days)")

In [ ]:
### 1.6.2: Memory 구성을 Parameter Store에 저장

from lab_helpers.parameter_store import put_parameter

# Lab 2-5에서 사용할 memory_id 저장
put_parameter(
    PARAMETER_PATHS["memory"]["memory_id"],
    memory_id,
    description="Memory ID for agent conversation history",
    region_name=AWS_REGION,
)

# Lab 2-4에서 사용할 기본 세션 ID 저장
put_parameter(
    PARAMETER_PATHS["memory"]["default_session_id"],
    "crm-session-id",
    description="Default session ID for Labs 2-4",
    region_name=AWS_REGION,
)

print("✅ PSM Keys stored:")
print(f"   • {PARAMETER_PATHS['memory']['memory_id']} = {memory_id}")
print(f"   • {PARAMETER_PATHS['memory']['default_session_id']} = crm-session-id")

### 요약: Memory 설정 완료

✅ **AgentCore Memory 리소스 생성 완료**
- 모든 Lab(2-5)을 위한 단일 공유 Memory 리소스
- 비용 관리를 위한 7일 후 자동 만료
- 멀티턴 대화 및 컨텍스트 로드 지원

✅ **Parameter Store 구성**
- Lab 2-5에서 조회할 수 있도록 Memory ID 저장
- Lab 2-4에서 기본 세션 ID 사용 가능
- 중앙 구성 패턴 준수

**다음 단계:**
- Lab 2: memory_id를 조회하고 Memory hook 초기화
- Lab 3-4: 복구/예방 Agent에 동일한 Memory 사용
- Lab 5: 공유 Memory를 사용한 멀티 Agent 오케스트레이션

## Part 2: 인프라 설정 및 CRM 애플리케이션 배포

인프라 설정 및 CRM 애플리케이션 배포는 워크숍 설정 과정에서 자동으로 수행됩니다.

다음 섹션으로 진행하세요.

In [ ]:
# 포트 80과 8080의 URL을 모두 시도
print(f"Click here to access the CRM App UI: '{get_app_url()}'")

## 1. 결함 주입 유틸리티 설정

이 섹션에서는 인프라 결함을 주입하는 도구를 준비하고 배포에 이미 포함된 사전 구성 결함을 검토합니다. 이 워크숍에는 종합적인 SRE 교육을 위한 **총 4개의 결함**이 포함되어 있습니다.

- **결함 1: DynamoDB 제한** - 테이블 용량을 줄여 ProvisionedThroughputExceededException 발생
- **결함 2: IAM 권한 문제** - EC2 역할 권한을 제한하여 AccessDenied 오류 발생

워크숍 전반에서 이러한 결함을 사용하여 다양한 장애 모드와 탐지 방법에 대한 SRE Agent의 진단 기능을 테스트합니다.

In [ ]:
from lab_helpers.lab_01.fault_injection import (
    initialize_fault_injection,
    inject_dynamodb_throttling,
    inject_iam_permissions,
)

# AWS 클라이언트를 초기화하고 SSM에서 인프라 리소스 ID 조회
print("Initializing fault injection utilities...")
resources = initialize_fault_injection(AWS_REGION, AWS_PROFILE)

print("\nDiscovered Infrastructure Resources:")
print(f"  Nginx Instance: {resources.get('nginx_instance_id', 'Not found')}")
print(f"  App Instance: {resources.get('app_instance_id', 'Not found')}")
print(f"  CRM Activities Table: {resources.get('crm_activities_table_name', 'Not found')}")
print(f"  CRM Customers Table: {resources.get('crm_customers_table_name', 'Not found')}")
print(f"  CRM Deals Table: {resources.get('crm_deals_table_name', 'Not found')}")
print(f"  EC2 Role: {resources.get('ec2_role_name', 'Not found')}")
print(f"  Public ALB DNS: {resources.get('public_alb_dns', 'Not found')}")

print("\n✅ Fault injection utilities ready")

## 2. 인프라 확인

결함을 주입하기 전에 CloudFormation 스택이 필요한 모든 리소스를 생성했으며 리소스가 정상 상태인지 확인합니다.

In [ ]:
from lab_helpers.lab_01.infrastructure import (
    verify_ec2_instances,
    verify_dynamodb_tables,
    verify_alb_health,
    verify_cloudwatch_logs,
)

print("Verifying infrastructure components...\n")

# EC2 인스턴스가 실행 중인지 확인
ec2_status = verify_ec2_instances(resources, AWS_REGION, AWS_PROFILE)

# DynamoDB 테이블이 존재하고 액세스 가능한지 확인
dynamodb_status = verify_dynamodb_tables(resources, AWS_REGION, AWS_PROFILE)

# ALB 대상이 정상 상태인지 확인
alb_status = verify_alb_health(resources, AWS_REGION, AWS_PROFILE)

# CloudWatch 로그 그룹이 존재하는지 확인
logs_status = verify_cloudwatch_logs(AWS_REGION, AWS_PROFILE)

if all([ec2_status, dynamodb_status, alb_status, logs_status]):
    print("\n✅ All infrastructure components verified and healthy")
else:
    print("\n⚠️  Some infrastructure components failed verification")

## 3. 결함 주입 테스트 및 사전 구성 결함 검토

이 섹션에서는 인프라 결함 2개를 주입하고 배포에 미리 구성된 추가 결함 2개를 검토합니다. 이 **4개의 결함**은 진단 Agent를 위한 종합적인 교육 시나리오를 제공합니다.


### 결함 1: DynamoDB 제한

**결함 설명:**
애플리케이션이 테이블에 프로비저닝된 읽기/쓰기 용량을 초과하면 DynamoDB 제한이 발생합니다. 이는 다음과 같은 경우에 발생할 수 있는 일반적인 프로덕션 문제입니다.
- 예기치 않은 트래픽 급증으로 프로비저닝된 용량 초과
- 테이블의 용량 단위가 부족하게 잘못 구성됨

**결함 주입 방법:**
`inject_dynamodb_throttling()` 헬퍼 함수는 다음 방법으로 이 문제를 시뮬레이션합니다.
- 지표 테이블의 결제 모드를 `PAY_PER_REQUEST`(무제한)에서 `PROVISIONED`로 변환
- 용량 한도를 매우 낮게 설정: **Read Capacity Unit 1개** 및 **Write Capacity Unit 1개**
- 이에 따라 테이블은 초당 약 1회의 읽기 및 1회의 쓰기 작업만 처리 가능
- 일반적인 애플리케이션 부하도 즉시 이 한도를 초과

**예상 영향:**
- 애플리케이션 로그에 `ProvisionedThroughputExceededException` 오류 발생
- 요청이 제한되고 재시도되면서 지연 시간 증가
- CloudWatch 지표에 제한된 요청 표시


In [ ]:
# DynamoDB 제한 결함 주입 실행
success = inject_dynamodb_throttling(resources, AWS_REGION, AWS_PROFILE)

if success:
    print("✅ DynamoDB throttling fault injected successfully")
    print("   → Table converted to PROVISIONED mode with 1 RCU/1 WCU")
    print("   → Normal application load will now trigger throttling")
else:
    print("❌ Failed to inject DynamoDB throttling fault")

### 애플리케이션 테이블에 부하 생성 

이제 엔드포인트에 부하 테스트를 수행합니다. 30초 동안 초당 20개의 동시 요청을 전송합니다. 이 부하는 크지 않지만 프로비저닝된 테이블 용량이 잘못 구성되어 있으므로 [애플리케이션 로그](https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsre-workshop$252Fcrm-application)에 `ProvisionedThroughputExceededException` 오류가 표시되고 CloudWatch 지표에 제한된 요청이 나타납니다. 부하 테스트 중 Customers 탭에 액세스하면 데이터 로드에 문제가 발생합니다.

In [ ]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor

alb_dns = resources["public_alb_dns"]
url = f"http://{alb_dns}:8080/api/customers"


def make_request(i):
    try:
        requests.get(url, timeout=5)
    except:
        pass


for second in range(1, 31):
    with ThreadPoolExecutor(max_workers=50) as executor:
        executor.map(make_request, range(50))

    if second % 10 == 0:
        print(f"Progress: {second}/30 seconds")

    time.sleep(1)

print("\n✓ Load test complete")

### 결함 2: IAM 권한 문제

**결함 설명:**
애플리케이션에 AWS 리소스 액세스에 필요한 권한이 없으면 IAM 권한 문제가 발생합니다. 이는 가장 일반적인 프로덕션 문제 중 하나이며, 흔히 다음과 같은 원인으로 발생합니다.
- 테스트 없이 지나치게 제한적인 보안 정책 적용
- 신뢰 정책 변경으로 인한 역할 수임 실패
- 보안 팀이 포괄적인 거부 정책 적용

**결함 주입 방법:**
`inject_iam_permissions()` 헬퍼 함수는 다음 방법으로 이 문제를 시뮬레이션합니다.
- 애플리케이션 서버에서 사용하는 EC2 인스턴스 IAM 역할 검색
- 원본 DynamoDB 액세스 정책 백업
- 주요 DynamoDB 작업에 대한 명시적 **Deny** 정책으로 교체
- 대상: `PutItem`, `GetItem`, `Query`, `Scan`, `UpdateItem`, `DeleteItem`
- Deny 정책이 Allow 정책보다 우선하므로 데이터베이스 액세스가 즉시 차단됨

**예상 영향:**
- 모든 데이터베이스 작업에 대해 애플리케이션 로그에 `AccessDenied` 예외 발생
- DynamoDB 액세스가 필요한 기능이 완전히 실패


In [ ]:
# IAM 권한 결함 주입 실행
success = inject_iam_permissions(resources, AWS_REGION, AWS_PROFILE)

if success:
    print("✅ IAM permission fault injected successfully")
    print(f"   → EC2 role '{resources.get('ec2_role_name', 'Unknown')}' now has Deny policy")
    print("   → All DynamoDB operations will return AccessDenied")
else:
    print("❌ Failed to inject IAM permission fault")

이제 애플리케이션을 호출할 때 어떤 응답이 반환되는지 테스트합니다. API가 DynamoDB에서 데이터를 조회하지 못하는 백엔드 문제로 인해 500 오류가 표시되어야 합니다.
**참고**: IAM 권한이 전파되는 데 1분 정도 걸릴 수 있습니다. 500 오류가 표시되지 않으면 잠시 기다린 후 다시 시도하세요.

In [ ]:
time.sleep(180)
alb_dns = resources["public_alb_dns"]

url = f"http://{alb_dns}:8080/api/deals"

print("\nGenerating 5 requests to trigger IAM errors...")
print(f"Target: {url}\n")

for i in range(10):
    try:
        response = requests.get(url, timeout=5)
        print(f"Request {i + 1} - Status: {response.status_code}")
    except Exception as e:
        print(f"Request {i + 1} - Error: {str(e)}")

    time.sleep(1)  # 과도한 부하를 방지하기 위한 짧은 지연

print("\n✓ Load complete - waiting 10 seconds for logs to propagate...")
time.sleep(10)

## 요약

✅ 사전 요구 사항을 확인하고 인프라를 배포했습니다. CRM 애플리케이션이 실행 중이며 CloudWatch를 통해 모니터링됩니다. Agent가 문제를 해결할 수 있도록 실제 프로덕션 문제를 시뮬레이션하는 결함을 주입했습니다.

다음: Lab 2 - 진단 Agent 구축(Lab-02-diagnostics-agent.ipynb)

In [ ]:
# 포트 80과 8080의 URL을 모두 시도
print(f"Click here to access the CRM App UI: '{get_app_url()}'")